[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quadrics/elliptic_physics/s2_physics.ipynb)

# Rigid bodies on the sphere

Ellipses on the 2-sphere that spin and collide. The rigid motions of the sphere are the rotations of space, so a body is placed by a rotor, and its shape is a quadric: a quadratic form whose negative points are the inside. Everything below is written in the algebra of three-dimensional space.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from dataclasses import replace

import numpy as np
from IPython.display import Image, display
from PIL.Image import fromarray

from numga.algebras import Spherical3D
from examples import instantiate
from examples.animation import save_animation
from examples.quadrics.elliptic_physics import render, scenarios

np.set_printoptions(precision=4, suppress=True)

ga = Spherical3D                                   # Cl(3): the points of the sphere are the directions of space
core = instantiate("examples.quadrics.elliptic_physics.core", ga)   # the physics engine, for this sphere
mv = core.mv

Point = ga.gatype.antivector()                     # a point of the sphere
Plane = ga.gatype.vector()                         # a great circle
Motor = ga.gatype.rotor()                          # a rigid motion of the sphere: a rotation of space
Quadric = ga.gatype((Plane, Point))                # primal: a point's polar plane
DualQuadric = ga.gatype((Point, Plane))            # dual: a plane's pole

basis = mv.antivector(np.eye(3)).normalized()      # the points on the x, y and z axes

A body's shape is an ellipse: a dual quadric that sums the axis points, each paired with itself, weighted by the squared tangents of the half-widths and −1 at the pole. Its inverse is the primal form, negative inside.

In the matrix language of conics the dual and primal conic matrices are inverse to each other up to scale, $C^* \propto C^{-1}$. Rendering is implicit: a pixel of the front hemisphere is a point of the sphere, and it is inside where the form is negative.

In [ ]:
half_widths = np.radians([30.0, 10.0])
# `basis & Plane` leaves the great circle open: three maps Scalar <- Plane, reading how a circle
# passes each axis point. Each axis point times its own reading is a dyad Point <- Plane, and
# the weighted sum of the three is the ellipse as a dual quadric: it sends a circle to its pole.
ellipse: DualQuadric = (basis * (basis & Plane) * np.array([*np.tan(half_widths) ** 2, -1.0])).sum(axis=0)   # [] Point <- Plane
placement: Motor = (mv.xy * 0.3).exp()                                  # [] Motor: turned in the view, centred on it
# Inverting the map gives the primal form, which sends a point to its polar circle. Like any map it
# is placed by pulling the point into the body's frame and pushing the circle back out:
form: Quadric = placement >> ellipse.inverse()(placement << Point)        # [] Plane <- Point; p ∨ form(p) < 0 inside

coordinates, disk, rim = render.hemisphere(480)                        # the front hemisphere's pixels, as (x, y, z)
pixels: Point = mv.yz * coordinates[:, 0] + mv.zx * coordinates[:, 1] + mv.xy * coordinates[:, 2]   # [n_pixels] Point
# The quadric's value at a point is the point joined with its own polar: a scalar per pixel.
inside = (pixels & form(pixels)) < 0.0                                 # [n_pixels] bool

# In Cl(3) the sphere's points are the bivectors and its great circles the vectors, so the printed
# type of Plane <- Point reads vector <- bivector:
print(form)

display(fromarray(render.paint(inside[None], render.rgb(["#38bdf8"]), disk, rim, 2)))

Two bodies are apart when some blend A + λB, λ > 0, of their forms is positive definite; in optimization this is the S-lemma. The least eigenvalue of the best blend is therefore a margin, negative when they collide, and its eigenvector is the deepest point. The contact wrench is pushed through each body's inverse inertia, and the elastic impulse reverses the rate at which the bodies close.

In [ ]:
# Two ellipses spinning toward each other. scenarios.ellipses fills each with mass points and forms its
# inertia, the map from a rate to its momentum: Σ m p ∨ (p × rate), each point joined with its own motion.
bodies = scenarios.ellipses(
    half_angles=np.array([[30.0, 8.0], [17.0, 17.0]]), mass=np.array([1.0, 1.2]),
    # 0.3 rad either side of the view's centre; the scenarios fold in a camera turn, which this undoes.
    placement=scenarios.CAMERA.inverse() * scenarios.toward(np.array([0.3, 0.3]), np.array([0.7 + np.pi, 0.7])),
    rate=np.array([[-0.5, -2.0, 0.3], [1.0, 2.0, 0.2]]), colors=["#38bdf8", "#f43f5e"], n_phi=96,
)
a, b = np.array([0]), np.array([1])

# bodies.C holds one primal form per body, a batch of maps; indexing picks one, like a batch of
# points. Detection, in the first body's frame: the margin of the best blend, and the deepest point.
relative: Motor = bodies.motor[a].inverse() * bodies.motor[b]                  # [pair] Motor
margin, deepest = core.overlap(bodies.C[a], relative >> bodies.C[b](relative << Point))   # [pair] Scalar, [pair] Point
contact_plane: Plane = bodies.C[a](deepest).normalized()              # [pair] Plane: the first body's polar plane at the deepest point
contact: Point = bodies.Q[a](contact_plane)                            # [pair] Point: its pole, the contact point
one = contact_plane | contact                                          # [pair] AntiBivector: the contact wrench in a's frame
other = relative << one                                                # [pair] AntiBivector: and in b's

# Handling: the rate at which the bodies close along the wrench, and the impulse that reverses it.
# Inverse inertia is a map from a wrench to the rate it produces, so applying it to the contact
# wrench gives each body's rate per unit push:
response_one, response_other = bodies.I_inv[a](one), bodies.I_inv[b](other)   # [pair] Bivector each
closing = response_one.regressive(bodies.momentum[a]) - response_other.regressive(bodies.momentum[b])
compliance = one.regressive(response_one) + other.regressive(response_other)
impulse = -2.0 * closing / compliance
momentum_a, momentum_b = bodies.momentum[a] + one * impulse, bodies.momentum[b] - other * impulse
closing_after = response_one.regressive(momentum_a) - response_other.regressive(momentum_b)

print("margin:", margin.to_array())
print("closing rate before and after:", closing.to_array(), closing_after.to_array())

inside = (pixels & bodies.world()[:, None](pixels)) < 0.0                 # [bodies, pixels] the same test, per body

display(fromarray(render.paint(inside, bodies.color, disk, rim, 2)))

The engine does this every substep for every pair within reach, after moving each body by a midpoint step along its rate. Seven ellipses, needles to discs:

In [ ]:
# Seven ellipses, needles to discs: half-widths in degrees, masses, where each starts, and body-frame rates.
bodies = scenarios.ellipses(
    half_angles=np.array([[30.0, 6.0], [17.0, 17.0], [28.0, 6.5], [24.0, 5.5], [15.0, 4.0], [26.0, 9.0], [9.0, 9.0]]),
    mass=np.array([1.0, 1.2, 0.9, 0.8, 0.5, 1.1, 0.4]),
    placement=scenarios.toward(np.array([0.35, 1.05, 1.15, 1.10, 1.25, 0.90, 1.55]), np.array([0.2, 0.7, 2.1, 3.6, 4.9, -0.67, 2.90])),
    rate=np.array([[0.8, 2.6, 0.5], [1.8, -1.2, 0.6], [-1.7, 1.5, -0.6], [2.0, 1.0, -0.5],
                   [-2.2, -1.8, 0.7], [1.6, 1.4, 0.7], [-2.4, 0.8, -0.4]]),
    colors=["#38bdf8", "#f43f5e", "#fbbf24", "#34d399", "#a855f7", "#fb923c", "#ec4899"],
    n_phi=384,
)
def step(bodies, dt: float):
    """A midpoint step of every body along its rate; the momentum stays in the body frame."""
    motor, momentum = bodies.motor, bodies.momentum
    half = (motor * (bodies.I_inv(momentum) * (dt / 4)).exp()).normalized()   # half a step ahead
    rate = bodies.I_inv((motor.inverse() * half) << momentum)                 # the rate at the midpoint
    moved = (motor * (rate * (dt / 2)).exp()).normalized()
    return replace(bodies, motor=moved, momentum=(motor.inverse() * moved) << momentum)


def collide(bodies, dt: float):
    """Elastic impulses for every pair that overlaps and is moving deeper in."""
    i, j = np.triu_indices(len(bodies.color), 1)                              # every pair once

    def margins(motors: Motor):
        relative = motors[i].inverse() * motors[j]
        return *core.overlap(bodies.C[i], relative >> bodies.C[j](relative << Point)), relative

    margin, deepest, relative = margins(bodies.motor)
    ahead = (bodies.motor * (bodies.rate() * (dt / 2)).exp()).normalized()    # every body a little further along
    touching = (margin < 0.0) & (margins(ahead)[0] < margin)                  # overlapping, and getting deeper
    contact_plane = bodies.C[i](deepest).normalized()
    wrench_one = contact_plane | bodies.Q[i](contact_plane)
    wrench_other = relative << wrench_one
    momentum = bodies.momentum
    for a, b, one, other in zip(i[touching], j[touching], wrench_one[touching], wrench_other[touching]):
        response_one, response_other = bodies.I_inv[a](one), bodies.I_inv[b](other)
        closing = response_one.regressive(momentum[a]) - response_other.regressive(momentum[b])
        impulse = -2.0 * closing / (one.regressive(response_one) + other.regressive(response_other))
        momentum = momentum.at[a].set(momentum[a] + one * impulse).at[b].set(momentum[b] - other * impulse)
    return replace(bodies, momentum=momentum)


def simulate(bodies, frames: int, dt: float):
    """Each frame's state, as it is asked for: six substeps of motion and contact between frames."""
    for frame in range(frames):
        yield bodies
        for substep in range(6):
            bodies = collide(step(bodies, dt / 6), dt / 6)


# Each frame, the implicit test per pixel and body, computed as the animation is written:
coverage = ((pixels & state.world()[:, None](pixels)) < 0.0 for state in simulate(bodies, 160, 0.015))
frames = (render.paint(inside, bodies.color, disk, rim, 2) for inside in coverage)

display(Image(filename=save_animation(frames, "spherical_quadric_physics", 15)))

In [ ]:
# checks
assert margin.to_array()[0] < 0.0                                      # the two ellipses meet
np.testing.assert_allclose(closing_after.to_array(), -closing.to_array(), rtol=1e-12)   # the closing rate reverses
states = list(simulate(bodies, 40, 0.015))                             # a shorter run of the same crowd
energy = np.array([state.kinetic_energy().to_array() for state in states])
momentum = np.array([state.total_momentum().norm().to_array() for state in states])
assert np.ptp(energy) / energy[0] < 1e-3                               # the crowd keeps its energy
assert np.ptp(momentum) / momentum[0] < 1e-12                          # and its total momentum